# Figures

## Load simulation data

In [ ]:
include("packages.jl")

filename = "output/sol_2025-11-19_14-39.jld2"

@load filename sol local_tags_list

2-element Vector{Symbol}:
 :sol
 :local_tags_list

## Translation elongation rate profile

## Divide u in multiple tables

In [ ]:

# Save results for each sequence in this set
function save_competition_results(sol, seq_names, seqs, local_tags_list, n_metab, csv_dir)
	
	idx = n_metab - 5
    
	for (seq_idx, seq_name) in enumerate(seq_names)
        local_tags = local_tags_list[seq_idx]

		# Prepare data frame 
        df = DataFrame()
        df.time = sol.t

		# Metabolites
        for i in 1:n_metab 
            var_name = i <= length(local_tags.species_and_sequences.Tag) ? local_tags.species_and_sequences.Tag[i] : "State_$i"
            df[!, var_name] = [u[i] for u in sol.u]
        end

		# Output variables 

		out_start 	= n_metab - 4
		out_end		= n_metab 

		L = 0

		for i in out_start:out_end
			var_name = i <= length(local_tags.species_and_sequences.Tag) ? local_tags.species_and_sequences.Tag[i] : "State_$i"
            df[!, var_name] = [u[i + L] for u in sol.u]
		end

		# Ribosome complexes

        L = length(seqs[seq_name].Codon) - 1
        ribo_start = out_end + 1
        ribo_end = out_end + L

        for i in 1:L
            df[!, "Ribosome_$(i)"] = [u[ribo_start-1+i] for u in sol.u]
        end

		csv_file = joinpath(csv_dir, "$(seq_name)_competition_solution_testset.csv")
        CSV.write(csv_file, df)
        println("     ✓ Saved: $csv_file")

        idx += L + 5

    end
end

save_competition_results(sol, seq_names, seqs, local_tags_list, 138, csv_dir)